In [ ]:
import kfp
from google.cloud import aiplatform
from kfp.v2 import dsl, compiler
from kfp.v2.dsl import (Artifact, ClassificationMetrics, Input, Metrics, Output, component)
from google.cloud import storage
import os
from datetime import datetime
from uuid import uuid4
import pytz


In [ ]:
def generar_run_id() -> str:
    timestamp = datetime.now(pytz.utc).strftime("%Y%m%dT%H%M%SZ")
    token = uuid4().hex[:12]
    return f"house-price-{timestamp}-{token}"

In [ ]:
AAAAMM = '202605'
run_id = generar_run_id()

In [ ]:
# Ruta proyectos
project_id = os.getenv("GCP_PROJECT_ID")

# Ruta Storage
pipeline_ruta = os.getenv("PIPELINE_ROOT")
bucket_name = os.getenv("GCP_BUCKET_NAME")


In [ ]:
@component(
    packages_to_install=[
        "google-cloud-storage",
        "google-cloud-bigquery",
        "python-dotenv",
        "pandas",
        "pytz",
        "pandas_gbq"
    ],
)
def filter_input(
    AAAAMM: str, 
    run_id: str,
    periodo: Output
):
    import os
    from google.cloud import bigquery
    from dotenv import load_dotenv 
    from datetime import datetime
    import pytz
    from google.api_core.exceptions import NotFound
    import pandas as pd
    import pandas_gbq

    load_dotenv()
    
    # Ruta proyectos
    project_id = os.getenv("GCP_PROJECT_ID")

    # Tablas BigQuery inputs
    project_id_input = os.getenv("GCP_PROJECT_ID_INPUT")
    dataset_id_input = os.getenv("BQ_DATASET_ID_INPUT")
    table_id_input = os.getenv("BQ_TABLE_ID_INPUT")

    # Tablas features
    dataset_id_features = os.getenv("BQ_DATASET_ID_FEATURES")
    table_id_features = os.getenv("BQ_TABLE_ID_FEATURES")

    # Tablas temporales
    dataset_temp = os.getenv("BQ_DATASET_ID_TEMP")
    table_temp_data_input = os.getenv("BQ_TABLE_ID_TEMP_DATA")

    # Carga de BigQuery
    client = bigquery.Client(project=project_id)

    path_bq_table = f"{project_id_input}.{dataset_id_input}.{table_id_input}"
    dfInput = client.query(
        f'''SELECT * FROM `{path_bq_table}` where periodo = '{AAAAMM}'
        '''
    ).to_dataframe()

    path_bq_table_feature = f"{project_id}.{dataset_id_features}.{table_id_features}"
    dfFeatures = client.query(
        f'''SELECT * FROM `{path_bq_table_feature}`
        '''
    ).to_dataframe()

    listFeatures = dfFeatures['features'].tolist()
    dfInputFeat = dfInput[['id','periodo'] + listFeatures].copy()
    ZPeru = pytz.timezone("America/Lima")
    fecha_carga = datetime.now(ZPeru)
    dfInputFeat['date_subida_local'] = fecha_carga.replace(tzinfo=None)
    dfInputFeat['date_subida_utc'] = fecha_carga.astimezone(pytz.UTC)
    

    dfInputFeat = dfInputFeat[['run_id','id','periodo'] + listFeatures + ['date_subida_local', 'date_subida_utc']]
    periodo = dfInputFeat['periodo'].iloc[0]

    # Validacion campos duplicados:
    dfInputFeat.groupby(['id','periodo']).size()



    path_bq_data_input = f"{project_id}.{dataset_temp}.{table_temp_data_input}"
    try:
        delete_query = f"""
            DELETE FROM `{path_bq_data_input}`
            WHERE periodo = '{periodo}'
        """
        delete_job = client.query(delete_query)
        delete_job.result()

        print(f"Registros previos eliminados para período {periodo}.")

    except NotFound:
        print("La tabla destino no existe aún; se creará durante la carga.")

    pandas_gbq.to_gbq(
        dfInputFeat,
        '.'.join(path_bq_data_input.split('.')[1:]),
        if_exists = 'append',
        project_id=project_id
    )
    print(f"Tabla cargada: {path_bq_data_input}")
    

In [ ]:
@component(
    packages_to_install=[
        "google-cloud-storage",
        "google-cloud-bigquery",
        "python-dotenv",
        "joblib",
        "pandas",
        "pytz",
        "pandas_gbq"
    ],
)
def transform_predict(
    periodo: str,
    periodo_out: Output
):

    import os
    from dotenv import load_dotenv
    from google.cloud import storage
    from google.cloud import bigquery
    from google.api_core.exceptions import NotFound
    import types
    import joblib
    import sys
    from io import BytesIO
    from datetime import datetime
    import pandas as pd
    import pytz
    import pandas_gbq


    load_dotenv()

    # Ruta proyectos
    project_id = os.getenv("GCP_PROJECT_ID")

    # Ruta Storage
    pipeline_ruta = os.getenv("PIPELINE_ROOT")
    bucket_name = os.getenv("GCP_BUCKET_NAME")
    model_ruta = os.getenv("MODEL_ROOT")

    # Tablas features
    dataset_id_features = os.getenv("BQ_DATASET_ID_FEATURES")

    # Tablas temporales
    dataset_temp = os.getenv("BQ_DATASET_ID_TEMP")
    table_temp_data_input = os.getenv("BQ_TABLE_ID_TEMP_DATA")

    # Tablas out
    table_temp_data_transf_input = os.getenv("BQ_TABLE_ID_TEMP_DATA_TRANSF")


    # Carga de Storage
    client = storage.Client(project=project_id)
    bucket = client.bucket(bucket_name)
    
    metadata_name = 'metadata_transformer.py'
    metadata_path = f"{pipeline_ruta}/{metadata_name}"

    blob = bucket.blob(metadata_path)
    module_code  = blob.download_as_text(encoding="utf-8")

    metadata_transformer = types.ModuleType('metadata_transformer')
    metadata_transformer.__file__ = "gs://.../metadata_transformer.py"
    sys.modules['metadata_transformer'] = metadata_transformer

    exec(compile(module_code, metadata_transformer.__file__, "exec"),
        metadata_transformer.__dict__)

    pipeline_name = 'Pipeline-Transformacion-Training.joblib'
    pipeline_path = f"{pipeline_ruta}/{pipeline_name}"

    blob = bucket.blob(pipeline_path)
    contenido_joblib  = blob.download_as_bytes()
    pipeline = joblib.load(BytesIO(contenido_joblib))

    model_name  = "Model-GradientBoostingRegressor.joblib"
    model_path = f"{model_ruta}/{model_name}"

    blob = bucket.blob(model_path)
    contenido_joblib  = blob.download_as_bytes()
    modelo = joblib.load(BytesIO(contenido_joblib))


    # Carga de BigQuery
    client = bigquery.Client(project=project_id)

    path_bq_data_input = f"{project_id}.{dataset_temp}.{table_temp_data_input}"
    dfInputFeat = client.query(
        f'''SELECT * FROM `{path_bq_data_input}` WHERE periodo = '{periodo}'
        '''
    ).to_dataframe()

    X_escalado = pipeline.transform(dfInputFeat)

    feature_order = pipeline.named_steps[
    "feature_engineering"
    ].get_feature_names_out()

    dfInputFeatTransf = pd.DataFrame(
    X_escalado,
    columns = [col + '_transf' for col in feature_order],
    index =  dfInputFeat.index
    )

    dfInputFeatTransf['saleprice'] = modelo.predict(dfInputFeatTransf)
    dfInputFeatTransf['id'] = dfInputFeat['id']

    dfOutput = dfInputFeat.drop(columns =['date_subida_local','date_subida_utc']).merge(
    dfInputFeatTransf, on = ['id'], how='left')

    ZPeru = pytz.timezone("America/Lima")
    fecha_carga = datetime.now(ZPeru)
    dfOutput['date_subida_local'] = fecha_carga.replace(tzinfo=None)
    dfOutput['date_subida_utc'] = fecha_carga.astimezone(pytz.UTC)


    path_bq_data_transf_input = f"{project_id}.{dataset_temp}.{table_temp_data_transf_input}"
    try:
        delete_query = f"""
            DELETE FROM `{path_bq_data_transf_input}`
            WHERE periodo = '{periodo}'
        """
        delete_job = client.query(delete_query)
        delete_job.result()
        print(f"Registros previos eliminados para período {periodo}.")

    except NotFound:
        print("La tabla destino no existe aún; se creará durante la carga.")

    pandas_gbq.to_gbq(
        dfOutput,
        '.'.join(path_bq_data_transf_input.split('.')[1:]),
        if_exists = 'append',
        project_id=project_id
    )

    table_id_auditoria = f"{project_id}.{dataset_id_features}.model_execution_audit"

    # Fecha/hora de la ejecución actual, en UTC
    fecha_actualizacion = datetime.now(ZPeru)
    df_auditoria = pd.DataFrame([{
        "periodo": periodo,
        "model_name": model_name,
        "model_path_gcs": f"gs://{bucket_name}/{model_path}",
        "pipeline_path_gcs": (
            f"gs://{bucket_name}/{model_ruta}/Pipeline-Transformacion-Training.joblib"
        ),
        "rows_processed": len(dfOutput),
        "updated_at": fecha_actualizacion,
    }])

    pandas_gbq.to_gbq(
        df_auditoria,
        '.'.join(table_id_auditoria.split('.')[1:]),
        if_exists = 'append',
        project_id=project_id
    )


In [ ]:
@component(
    packages_to_install=[
        "google-cloud-bigquery",
        "python-dotenv",
        "pandas",
        "pytz",
        "pandas_gbq"
    ],
)
def send_out(
    periodo: str
):

    import os
    from google.cloud import bigquery
    from google.api_core.exceptions import NotFound
    from dotenv import load_dotenv
    import pandas_gbq
    import pandas as pd


    load_dotenv()
    
    # Ruta proyectos
    project_id = os.getenv("GCP_PROJECT_ID")

    # Tablas features
    dataset_id_features = os.getenv("BQ_DATASET_ID_FEATURES")
    table_id_features = os.getenv("BQ_TABLE_ID_FEATURES")

    # Tablas inputs
    dataset_temp = os.getenv("BQ_DATASET_ID_TEMP")
    table_temp_data_transf_input = os.getenv("BQ_TABLE_ID_TEMP_DATA_TRANSF")

    # Tablas Ouputs
    project_id_out = os.getenv("GCP_PROJECT_ID_OUT")
    dataset_id_out = os.getenv("BQ_DATASET_ID_OUT")
    table_out = os.getenv("BQ_TABLE_ID_OUT")
    table_out_hist = os.getenv("BQ_TABLE_ID_OUT_HIST")

    # Carga de BigQuery
    client = bigquery.Client(project=project_id)

    path_bq_table_feature = f"{project_id}.{dataset_id_features}.{table_id_features}"
    dfFeatures = client.query(
            f'''SELECT * FROM `{path_bq_table_feature}`
            '''
        ).to_dataframe()

    listFeatures = dfFeatures['features'].tolist()

    path_bq_data_transf_input = f"{project_id}.{dataset_temp}.{table_temp_data_transf_input}"
    dfOutput = client.query(
            f'''SELECT * FROM `{path_bq_data_transf_input}` WHERE periodo = '{periodo}'
            '''
    ).to_dataframe()


    listOut = ['id','periodo'] + listFeatures + ['saleprice','date_subida_local','date_subida_utc']
    dfOut = dfOutput[listOut].copy()

    table_id_out = f'{project_id_out}.{dataset_id_out}.{table_out}'
    delete_query = f"""
        TRUNCATE TABLE `{table_id_out}`
    """

    delete_job = client.query(delete_query)
    delete_job.result()

    pandas_gbq.to_gbq(
        dfOut[listOut],
        '.'.join(table_id_out.split('.')[1:]),
        if_exists = 'append',
        project_id=project_id_out
    )

    table_id_out_hist = f'{project_id_out}.{dataset_id_out}.{table_out_hist}'
    try:
        delete_query = f"""
            DELETE FROM `{table_id_out_hist}`
            WHERE periodo = '{periodo}'
        """
        delete_job = client.query(delete_query)
        delete_job.result()

        print(f"Registros previos eliminados para período {periodo}.")

    except NotFound:
        print("La tabla destino no existe aún; se creará durante la carga.")

    pandas_gbq.to_gbq(
        dfOut[listOut],
        '.'.join(table_id_out_hist.split('.')[1:]),
        if_exists = 'append',
        project_id=project_id_out
    )



In [ ]:
@kfp.dsl.pipeline(
    name="pipeline-data-processing-predict", 
    description="Proyecto de pipeline para data processing y predict",
    pipeline_root="gs://gcp-processing-storage-prod/project-pipeline-predictions-casas"
)
def main_pipeline(
    AAAAMM: str, 
    run_id: str
):



    filter_input = filter_input(
        AAAAMM = AAAAMM
    )
    filter_input.set_display_name("FILTER DATA INPUT")

    transform_predict = transform_predict(
        periodo = filter_input.output
    ).after(filter_input)
    transform_predict.set_display_name("TRANSFORM PREDICT")

    send_out = send_out(
        periodo = filter_input.output
    ).after(transform_predict)
    send_out.set_display_name("SEND OUTPUT")



In [ ]:
compiler.Compiler().compile(
    pipeline_func=main_pipeline,
    package_path="pipeline-data-processing-predict.json"
)

In [ ]:
bucket_name

In [ ]:
def upload_to_gcs(project_id, bucket_name, source_file_name, destination_blob_name):
    storage_client = storage.Client(project_id = project_id)
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(source_file_name)
    print(f"Archivo {source_file_name} subido a {destination_blob_name} en el bucket {bucket_name}.")


destination_blob_name = "demo/pipeline_prediction.json"
pipeline_file = "pipeline_prediction.json"
upload_to_gcs(bucket_name, pipeline_file, destination_blob_name)

In [ ]:
aiplatform.init(project="<project_id>", location="us-central1")

job = aiplatform.PipelineJob(
    display_name="pipeline de prueba",
    template_path="gs://<bucket>/demo/pipeline_prediction.json",
    enable_caching=False,
    project="<project_id>",
    location="us-central1",
    parameter_values={
                        "":""
                     }
    #labels={"module": "ml", "application": "app", "chapter": "mlops", "company": "datapat", "environment": "dev", "owner": "xxxx"}
)

print('submit pipeline job ...')
job.submit(service_account="")